<a href="https://colab.research.google.com/github/rohitvmeshram/Fetching-and-Analyzing-Top-50-Live-Cryptocurrency-Data/blob/main/Fetching_and_Analyzing_Top_50_Live_Cryptocurrency_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import time
from openpyxl import load_workbook

# API Endpoint (Using CoinGecko for simplicity)
API_URL = "https://api.coingecko.com/api/v3/coins/markets"
PARAMS = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 50,
    "page": 1,
    "sparkline": False
}

def fetch_crypto_data():
    """Fetch live cryptocurrency data from CoinGecko."""
    response = requests.get(API_URL, params=PARAMS)
    if response.status_code == 200:
        return response.json()
    else:
        print("Error fetching data", response.status_code)
        return None

def process_data(data):
    """Process the data and perform analysis."""
    # Create a DataFrame with only the required columns
    df = pd.DataFrame(data, columns=[
        "name",
        "symbol",
        "current_price",
        "market_cap",
        "total_volume",
        "price_change_percentage_24h"
    ])

    # Identify the top 5 cryptocurrencies by market cap
    top_5 = df.nlargest(5, "market_cap")

    # Calculate the average price of the top 50 cryptocurrencies
    avg_price = df["current_price"].mean()

    # Find the highest and lowest 24-hour percentage price change
    highest_change = df.nlargest(1, "price_change_percentage_24h")
    lowest_change = df.nsmallest(1, "price_change_percentage_24h")

    return df, top_5, avg_price, highest_change, lowest_change

def write_to_excel(df):
    """
    Write the DataFrame to an Excel file.

    Note: Excel (.xlsx) files are binary files. Do not attempt to read them as UTF-8 encoded text.
    """
    filename = "crypto_data.xlsx"
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name="Live Data")
    print(f"Data written to {filename}")

def main():
    while True:
        data = fetch_crypto_data()
        if data:
            df, top_5, avg_price, highest_change, lowest_change = process_data(data)
            write_to_excel(df)

            # Build a comprehensive report in one single print statement
            analysis_report = f"""\
-------------------------------------------------------
Top 5 Cryptocurrencies by Market Cap:
{top_5.to_string(index=False)}

-------------------------------------------------------
Average Price of Top 50 Cryptocurrencies: ${avg_price:.2f}

-------------------------------------------------------
Highest 24h Percentage Change:
{highest_change.to_string(index=False)}

-------------------------------------------------------
Lowest 24h Percentage Change:
{lowest_change.to_string(index=False)}

-------------------------------------------------------
Updating in 5 minutes...
"""
            print(analysis_report)

        # Wait 5 minutes before updating
        time.sleep(300)

if __name__ == "__main__":
    main()


Data written to crypto_data.xlsx
-------------------------------------------------------
Top 5 Cryptocurrencies by Market Cap:
    name symbol  current_price    market_cap  total_volume  price_change_percentage_24h
 Bitcoin    btc   94985.000000 1884974904900   38869945612                     -1.98717
Ethereum    eth    2584.610000  311459067133   23914078309                     -2.74354
  Tether   usdt       0.999917  141941958565   78158720386                     -0.01797
     XRP    xrp       2.380000  137594259162    4893516891                     -3.39851
     BNB    bnb     644.930000   94129568538    1436048015                      1.77941

-------------------------------------------------------
Average Price of Top 50 Cryptocurrencies: $4117.67

-------------------------------------------------------
Highest 24h Percentage Change:
name symbol  current_price  market_cap  total_volume  price_change_percentage_24h
 BNB    bnb         644.93 94129568538    1436048015               

KeyboardInterrupt: 